# Tarea 3 — Vbak_Silver_Merge

Anti-join contra `vbak_silver`: inserta solo las órdenes que aún no existen (por `VBELN`).
Depende de: `Vbak_Autoloader_Ingesta`

In [0]:
from pyspark.sql.functions import col, to_date, current_date

CATALOG = "laboratory_sap_dev"
SCHEMA  = "sap_course"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

nuevas = (
    spark.table(f"{CATALOG}.{SCHEMA}.vbak_bronze_autoloader").alias("b")
    .join(
        spark.table(f"{CATALOG}.{SCHEMA}.vbak_silver").alias("s"),
        col("b.VBELN") == col("s.VBELN"),
        "left_anti"
    )
    .select(
        col("VBELN"),
        to_date(col("ERDAT"), "yyyyMMdd").alias("ERDAT"),
        col("AUART"),
        col("KUNNR"),
        col("VKORG"),
        col("NETWR").cast("double"),
        col("WAERK"),
        current_date().alias("updated_at")
    )
)

n = nuevas.count()
if n > 0:
    nuevas.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.vbak_silver")
    print(f"OK: {n} ordenes nuevas insertadas en vbak_silver")
else:
    print("Sin ordenes nuevas - vbak_silver ya esta al dia")